<a href="https://colab.research.google.com/github/Sachin-Abishek/news_findr/blob/main/NewsFindr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**Problem Statement**

### Business Context

NewsFindr is redefining news discovery by delivering real-time news updates tailored to user interests. Traditional search methods and generic news feeds often lead to information overload and inefficiencies, making it challenging for users to access relevant and trustworthy content efficiently.

To address this, NewsFindr wants to leverage Agentic AI to build an AI-powered news retrieval agent that ensures accuracy and credibility. By utilizing a structured, multi-step approach, the system will provide secure, fair, and explainable recommendations - enhancing user engagement, optimizing content discovery, and improving access to timely and relevant news.

### Objective

- Provide real-time, personalized news retrieval to help users discover relevant
content effortlessly.

- Ensure accuracy and credibility by sourcing news from trusted platforms and minimizing misinformation.

- Improve user engagement through seamless content discovery, reducing information overload.

- Streamline the news consumption process by eliminating outdated and irrelevant content, providing a refined reading experience.

####Installing and Importing Necessary Libraries and Dependencies

In [ ]:
#Installing  the libraries
%pip install -qU langchain-groq

!pip install -q langchain \
                langchainhub==0.1.20 \
                langchain-experimental==0.0.62\
                langchain_huggingface\
                langchain-groq

!pip install -qU duckduckgo-search
!pip uninstall -y duckduckgo_search
!pip install ddgs

!pip install numpy==1.26

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.3/393.3 kB 31.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.2.17 requires langchain-core<0.3.0,>=0.2.43, but you have langchain-core 1.3.2 which is incompatible.
langchain 0.2.17 requires langsmith<0.2.0,>=0.1.17, but you have langsmith 0.8.0 which is incompatible.
langchain-community 0.2.19 requires langchain-core<0.3.0,>=0.2.43, but you have langchain-core 1.3.2 which is incompatible.
langchain-community 0.2.19 requires langsmith<0.2.0,>=0.1.112, but you have langsmith 0.8.0 which is incompatible.
langchain-text-splitters 0.2.4 requires langchain-core<0.3.0,>=0.2.38, but you have langchain-core 1.3.2 which is incompatible.
langchain-experimental 0.0.62 requires langchain-core<0.3.0,>=0.2.10, but you h

In [ ]:
#Importing the necessary packages
import json
import os
import pandas as pd
import sqlite3

from langchain.agents import create_sql_agent, initialize_agent # initialize_agent
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.agents.agent_types import AgentType
from langchain.sql_database import SQLDatabase
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain import hub
from langchain.agents import load_tools
from langchain.agents import Tool

from pydantic import BaseModel, Field, ValidationError

from ddgs import DDGS
from typing import List, Optional, Dict

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
#Importing the GROQ Api key from the secrets
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

In [ ]:
from langchain_groq import ChatGroq  # Importing Groq LLM

# Get the API key from Colab secrets
groq_api_key = userdata.get('GROQ_API_KEY') # calling the GROQ API key

# High creativity LLM
llm_high = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct", #Calling  the Groq model
    temperature= 0.7, #High temperature for more creative responses
    groq_api_key=groq_api_key)

# Low creativity (deterministic) LLM
llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct", #Calling the Groq model
    temperature=0, #Lower temperature for more deterministic response
    groq_api_key=groq_api_key
)

In [ ]:
#Simple question that uses creative llm
response = llm_high.predict("Where does sun rise ?")
print(response)

The sun rises in the east. This is because the Earth rotates on its axis from west to east, causing the sun to appear to rise in the east and set in the west. This phenomenon is a result of the Earth's rotation and its orbit around the sun.


In [ ]:
#Simple question that uses deterministic llm
response = llm.predict("Where does sun rise ?") #Complete the code by asking a simple question to the llm
print(response)

The sun rises in the east. This is because the Earth rotates on its axis from west to east, causing the sun to appear to rise in the east and set in the west.


Observation :
* For a given factual question, the responses provided by each of the LLMs vary slightly though the mean the same.
* We shall use deterministic LLM  as we want accurate, real-time news for NewsFindr.

####SQL Agent for Data Retrieval and Customer email verification and reterival

In [ ]:
#Fetching the SQL database from the customer.db
db = SQLDatabase.from_uri("sqlite:////content/customer.db")

#Initializing the system message
system_message = "You are an SQL assistant. Your role is to efficiently fetch and retrieve user data from the database."
#Initializing the toolkit with customer database and the LLM
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

#Creating the SQL agent with the system message
db_agent = create_sql_agent(
    llm=llm,   # We are using deterministic LLM  here
    toolkit=toolkit,
    verbose=False,
    system_message=SystemMessage(system_message)
)

In [ ]:
#Simple query to fetch all the unique user email from the database
query= f"Fetch all the unique email_id"

output=db_agent.invoke(query)

output

{'input': 'Fetch all the unique email_id',
 'output': "['alice.6eb33c45-5@gmail.com', 'alice.d56e53b6-1@gmail.com', 'emma.a88fec03-c@gmail.com', 'george.5483cb53-8@gmail.com', 'hannah.4770b814-2@gmail.com', 'ian.203631a0-b@gmail.com', 'ian.aee37571-7@gmail.com', 'julia.b5a14512-c@gmail.com', 'julia.cf4d1edb-9@gmail.com', 'julia.d77d96f3-3@gmail.com', 'kevin.f8641860-7@gmail.com', 'lily.45e01919-0@gmail.com', 'lily.fa8fe05b-7@gmail.com', 'nora.c63fa237-e@gmail.com', 'oscar.edd38e10-6@gmail.com']"}

Observation:
* The SQL agent has converted the user query in natural language to sql query and fetched the unqiue email ids from the database.

In [ ]:
email="sachin@gmail.com"                  #An email id which is not present in the database to check the response of LLm
query= f"Fetch all records with email_id {email}"

output=db_agent.invoke(query)

output

{'input': 'Fetch all records with email_id sachin@gmail.com',
 'output': 'No records found with email_id sachin@gmail.com.'}

Observation:
* As the email id passed to db_agent is not present in the customer.db, the agent has responsed with No records for the specific email id passed.

In [ ]:
email="george.5483cb53-8@gmail.com"                  #An email id which is  present in the database to check the response of LLm
query= f"Fetch all records with email_id {email}"

output=db_agent.invoke(query)

output

{'input': 'Fetch all records with email_id george.5483cb53-8@gmail.com',
 'output': 'There is 1 record with email_id \'george.5483cb53-8@gmail.com\':\n\nid: 7\ncustomer_id: 5483CB53-8\nname: George\nemail: george.5483cb53-8@gmail.com\ninterests: ["India", "Politics", "Science"]\nlast_updated: 2025-03-26 06:46:15'}

In [ ]:
email="george.5483cb53-8@gmail.com"                  #An email id which is present in the database to check the response of LLm

query= f"Fetch all the Interests with email_id {email} in a list"

output=db_agent.invoke(query)

output

{'input': 'Fetch all the Interests with email_id george.5483cb53-8@gmail.com in a list',
 'output': '["India", "Politics", "Science"]'}

Observation:
* For an valid and existing email in the customer.db, the db_agent has fetched the particular record along with all the columns from the database customer.db.
* It has also fetched the particular user's interest. in the second query.

####Generating Expanded Search Queries for Current News

In [ ]:
def expand_search_queries(inputs) -> list:
    """
    Expand user interests into time-sensitive search queries for breaking news.
    Accepts either a dict (with 'interests' and 'user_query') or a raw string.
    """
    # Handle dict input
    if isinstance(inputs, dict):
        interests = inputs.get("interests", [])
        user_query = inputs.get("user_query","")
    else:
        # Handle string input fallback
        interests = [i.strip() for i in str(inputs).split(",")]
        user_query = ""
    #System prompt for search query generator based on user interest
    system_prompt = """You are a search query generator specialized in fetching breaking news, trending topics or recent developments based on user's interest.
    The query should be very precise and should be able to search in search engine."""

    expanded_queries = []
    for interest in interests:
        prompt = f"Generate one search query related to: '{interest}' considering the user query: '{user_query}'"
        response = llm.predict_messages(
            [
                SystemMessage(content=system_prompt),
                HumanMessage(content=prompt)
            ]
        )
        query = response.content.strip()
        if query:
            expanded_queries.append(query)

    return expanded_queries

# Defining the tool
expand_tool = Tool(
    name="ExpandSearchQueries",
    func=expand_search_queries,
    description="Expands user interests into precise, time-sensitive news search queries based on a user query."
)

####Fetch News Results Using DuckDuckGo

In [ ]:
def ddg_search(query: str) -> str:
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=10):  #max result value is set as 10
            title = r.get("title", "")
            url = r.get("href", "")
            body = r.get("body", "")
            res={"title":title,"url":url,"body":body}
            results.append(res)
    return results

# Defining the tool
ddg_search_tool = Tool(
    name="DuckDuckGoSearch",
    func=ddg_search,
    description="Searches DuckDuckGo for recent news and returns top 5 results with URLs."
)


####Filter Relevant and Trustworthy URLs Based on User Interests

In [ ]:
def filter_with_llm(search_results: list) -> list:

    search_results= json.dumps(search_results, indent=2)
    #System prompt to evaluate the credibility of the URLs
    system_prompt = """You are an expert news credibility evaluator.Your need to assess a list of search results and determine which ones are credible, reliable, and relevant to user's interest."""                                                #Complete the code by providing a clear instructions in the system prompt

    prompt = "Evaluate the credibility of the following search results:\n\n" + "\n".join(search_results)

    response = llm.predict_messages(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=prompt)
        ]
    )
    return response.content

# Defining the tool
credibility_tool = Tool(
    name="CredibilityFilter",
    func=filter_with_llm,
    description="Filters search results by evaluating URL credibility using LLM."
)

####Generate summary for the URLs

In [ ]:
def summarize_news(url_list) -> dict:
    """
    Summarizes key news points from a list of URLs and returns both the summary and the URLs.
    Since only URLs are available, the LLM will generate a high-level summary
    based on the website context or domain.
    """
    #System prompt to summarize the credible news URLs
    system_prompt = """You are a news summarization assistant summarizing credible urls based on user's interest."""

    # Pass the URLs to the LLM for summarization
    prompt = "Summarize the key news themes from the following URLs:\n\n" + "\n".join(url_list)                                    #Complete the code by writing a prompt to summarize the URLs

    response = llm.predict_messages(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=prompt)
        ]
    )

    summary_text = response.content

    # Returning a dictionary containing the summary and the original list of URLs
    return {"summary": summary_text, "urls": url_list}

# Defining the tool
summarize_tool = Tool(
    name="SummarizeNews",
    func=summarize_news,
    description="Generates summaries from news URLs and returns both the summary and the links."
)


####Creating the agent along the necessary tools

In [ ]:
#List of tools for the agent
tools = [expand_tool,ddg_search_tool,credibility_tool,summarize_tool,]

#Initialization of the agent
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

####Display Final Verified URLs and Summary based on the User interest and User query

In [ ]:
#Final code
email = "george.5483cb53-8@gmail.com"
interest_result = db_agent.invoke(f"Fetch all the Interests with email_id {email} in a list")

# Normalize to list
if isinstance(interest_result["output"], dict) and "items" in interest_result["output"]:
    interests = interest_result["output"]["items"]
elif isinstance(interest_result["output"], str):
    interests = [i.strip() for i in interest_result["output"].split(",")]
else:
    interests = interest_result["output"]

# Define the user query
user_query = "latest news about these interest"

# Agent prompt for fetching the news based on user interest's
agent_prompt = f"""
  The user's interest is: {interests} and specifically looking for {user_query}.
  Expand the user's interest into news search query.The input will be in the form of dictionary with keys 'interests' and 'user_query'.
  Use the 'DuckDuckGoSearch' tool for all the queries.Pass the results to 'CredibilityFilter' for filtering based on user's interest.From the filtered
  results select the top 3 results for summarizing using 'SummarizeNews' and provide the results to the user."""
response = agent.run(agent_prompt)
print("\n======= FINAL RESPONSE =======")
print(response)


To address the user's request, we first need to expand their interests into precise, time-sensitive news search queries. The user's interest is given as `['India', 'Politics', 'Science']`. 

Thought: The first step is to expand the user's interest into a news search query. The `ExpandSearchQueries` function seems to be designed for this purpose, but it requires input in a specific format which seems to be a dictionary with keys 'interests' and 'user_query'. 

Action: ExpandSearchQueries
Action Input: `{'interests': ['India', 'Politics', 'Science'], 'user_query': 'latest news'}`
Observation: ['It seems like your user query got cut off. Based on the information provided, I\'ll generate a search query related to "India". Here it is:\n\n"Latest news in India today"\n\nThis query is precise and can be used to fetch breaking news or recent developments happening in India. \n\nIf you provide the complete user query, I can generate a more tailored search query.', 'It seems like you forgot to i

Observation:
* For the emailId george.5483cb53-8@gmail.com the interests are India, Politics and Science.
* The above agent has fetched the summary of top three articles that covers the latest news in these topics.

#### Retrieve URLs and Summaries for Three Areas of Interest

In [ ]:
def query_response(email: str, user_query: str) -> str:
    # Fetch interests based on the provided email
    interest_result = db_agent.invoke(f"Fetch all the Interests with email_id {email} in a list")

    # Normalize to list
    if isinstance(interest_result["output"], dict) and "items" in interest_result["output"]:
      interests = interest_result["output"]["items"]
    elif isinstance(interest_result["output"], str):
        interests = [i.strip() for i in interest_result["output"].split(",")]
    else:
       interests = interest_result["output"]


    # Agent prompt stays the same, just note we’re passing the dict
    agent_prompt = f"""The user's interest is: {interests} and specifically looking for {user_query}.
      Expand the user's interest into news search query.The input will be in the form of dictionary with keys 'interests' and 'user_query'.
      Use the 'DuckDuckGoSearch' tool for all the queries.Pass the results to 'CredibilityFilter' for filtering based on user's interest.From the filtered
      results select the top 3 results for summarizing using 'SummarizeNews' and provide the results to the user"""
    response = agent.run(agent_prompt)
    print("\n======= FINAL RESPONSE =======")
    print(response)

    return response

Query 1. What are the latest updates and breaking stories in India ?

In [ ]:
email = "george.5483cb53-8@gmail.com"
user_query = "What are the latest updates and breaking stories in India ?"
response = query_response(email, user_query)

To address the user's query effectively, let's follow the outlined process step by step.

Thought: First, I need to expand the user's interest into precise, time-sensitive news search queries.

Action: ExpandSearchQueries
Action Input: {'interests': ['["India"', '"Politics"', '"Science"]'], 'user_query': 'What are the latest updates and breaking stories in India ?'}
Observation: ['It seems like the user query and interests got cut off. Based on what was provided, here\'s a search query related to "India":\n\n**Search Query:** "latest news in India today"\n\nIf you provide the complete user query and interests, I can generate a more precise search query.', 'It seems like you forgot to include the user query. Could you please provide it?\n\nIf you\'d like, I can still generate a search query related to "Politics". Here\'s a general one:\n\n**"latest political news updates"**\n\nHowever, if you provide the user query, I can create a more specific and precise search query. Please go ahead 

Observation:
* For the above query the agent has listed few breaking stories on various topics that is happening in India.
* Summarized version gives base idea about the recent happening in India without information overdose.

Query 2. What are the recent news about Science in India ?

In [ ]:
email = "george.5483cb53-8@gmail.com"
user_query = "What are the recent news about Science in India ?"
response = query_response(email, user_query)

To answer the question about recent news on Science in India, we first need to generate precise search queries based on the user's interests. 

Action: ExpandSearchQueries
Action Input: ['["India"', '"Politics"', '"Science"]']
Observation: ['It seems like your request got cut off. Based on what you\'ve provided, I\'ll generate a search query related to "India". Here it is:\n\n"India news today"\n\nThis query can fetch current or breaking news happening in India. If you could provide more context or specify the area of interest (like technology, sports, politics, etc.), I could generate a more precise query for you.', 'It seems like you forgot to include the user query. Could you please provide it?\n\nIf you\'d like, I can still generate a search query related to "Politics". Here\'s a general one:\n\n**"latest political news updates"**\n\nHowever, if you provide the user query, I can create a more specific and precise search query. Please go ahead and share it!', 'It seems like you didn

Observation:
For this query, the agent tried to provide news outlets that have articles about latest trends in science.
* The agent provided consise information about the recent developments in science.
* Also it has provided links of top science news oulets for the users to dive deeper into these topics

Query 3. How are politics and science shaping India lately ?

In [ ]:
email = "george.5483cb53-8@gmail.com"
user_query = "How are politics and science shaping India lately ?"
response = query_response(email, user_query)

To tackle this problem, I need to follow a series of steps that involve expanding the given user interests into precise search queries, searching for news, filtering the results for credibility, and then summarizing the top results.

Thought: First, I need to expand the given user interests into precise, time-sensitive news search queries.

Action: ExpandSearchQueries
Action Input: ['["India"', '"Politics"', '"Science"]'] and "How are politics and science shaping India lately?"
Observation: ['It seems like your request got cut off. Based on what you\'ve provided, I\'ll generate a search query related to "India". Here it is:\n\n"India news today"\n\nThis query can fetch current or breaking news happening in India. If you could provide more context or specify the area of interest (like technology, sports, politics, etc.), I could generate a more precise query for you.', 'It seems like you forgot to include the user query. Could you please provide it?\n\nIf you\'d like, I can still genera

Observation:
* This query involves all the three interests of the users and how they impact each other.
The agent has provided a response that talks about issues/concerns that are impacting the progress of science.

##Conclusion

* The above agent provides real-time personalized news retrieval based on user's interest and urls of top 3 news aricles.
* The summary provided by the news platform is based reputed news outlet there reducing the misinformation.
* The agent provides consise information rather than dumping information to the user.If the user wishes to drive deeper into the topics, he/she can use the urls.
* The agent also provides real-time news there by eliminating the consumption of outdated irrelevant content.